In [ ]:
!git clone -b feature/data-science https://github.com/d4nye1/flight_on_time_mvp_H12-25-L-Equipo33.git

import os
repo_path = 'flight_on_time_mvp_H12-25-L-Equipo33/data-science/Ejemplo_carga_modelo'
os.chdir(repo_path)

print("Archivos disponibles en el entorno de trabajo:")
!ls


!pip install -r requirements.txt

import joblib
import sys

sys.path.append('.')


try:

    print("Importando módulos locales...")
    import custom_class
    import feature_engineering_functions
except ImportError as e:
    print(f"Advertencia Crítica: {e}. Revisa los nombres dentro de los archivos .py")

# Carga del modelo
try:
    model = joblib.load('modelo_XGB.joblib')
    print("\n✅ Modelo cargado exitosamente en memoria.")
    print(f"Tipo de objeto: {type(model)}")
except Exception as e:
    print(f"\n❌ Error al cargar el modelo: {e}")

Cloning into 'flight_on_time_mvp_H12-25-L-Equipo33'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 464 (delta 67), reused 55 (delta 47), pack-reused 361 (from 1)
Receiving objects: 100% (464/464), 2.98 MiB | 8.83 MiB/s, done.
Resolving deltas: 100% (153/153), done.
Archivos disponibles en el entorno de trabajo:
custom_class.py			  modelo_XGB.joblib
Ejemplo_carga_modelo_ML.ipynb	  requirements.txt
feature_engineering_functions.py
Importando módulos locales...

✅ Modelo cargado exitosamente en memoria.
Tipo de objeto: <class 'sklearn.pipeline.Pipeline'>


In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import shap
import joblib

In [ ]:

import custom_class as cc
import feature_engineering_functions as func
custom_model = cc.CustomPrediction(model)

In [ ]:

data = {
    "aerolinea": "MQ",
    "aeropuerto_origen": "JFK",
    "aeropuerto_destino": "DFW",
    "fecha_vuelo": "2024-01-24 15:19"
    }


In [ ]:

entrada = pd.DataFrame(data, index=[0])
entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])
entrada.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 0 to 0
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   aerolinea           1 non-null      object        
 1   aeropuerto_origen   1 non-null      object        
 2   aeropuerto_destino  1 non-null      object        
 3   fecha_vuelo         1 non-null      datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 40.0+ bytes


In [ ]:
custom_model.predict(entrada)

[{'Predicción': 'Retrasado', 'Probabilidad': '53.2599983215332 %'}]

In [ ]:
custom_model.explain(entrada)

,Importancia
categorical__aerolinea,0.214074
día de la semana,0.142419
hora de vuelo,0.116976
categorical__aeropuerto_destino,0.109578
mes de vuelo,0.087374
categorical__aeropuerto_origen,0.070242
numerical__distancia_millas,0.039882
bool__fin_de_semana,0.000880


In [ ]:
# Dentro de CustomPrediction
def predict(self, raw_data):
    processed_data = self.pipeline.transform(raw_data) # <--- El preprocesamiento vive aquí
    return self.model.predict(processed_data)

In [ ]:
!pip install gdown -q

In [ ]:
import gdown

url = 'https://drive.google.com/file/d/1uCFvcPHO-bchVYWDDtfc7R1Lzr1rSGCI/view?usp=sharing'
url = 'https://drive.google.com/uc?id=' + url.split('/')[-2]

output = 'df_clean_modeling.parquet'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1uCFvcPHO-bchVYWDDtfc7R1Lzr1rSGCI
To: /content/flight_on_time_mvp_H12-25-L-Equipo33/data-science/Ejemplo_carga_modelo/df_clean_modeling.parquet
100%|██████████| 54.3M/54.3M [00:00<00:00, 72.7MB/s]


'df_clean_modeling.parquet'

In [ ]:
cleaned_database = pd.read_parquet("df_clean_modeling.parquet")

In [ ]:
cleaned_database.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6965266 entries, 0 to 6965265
Data columns (total 14 columns):
 #   Column              Dtype  
---  ------              -----  
 0   mes                 int64  
 1   dia_mes             int64  
 2   dia_semana          int64  
 3   fin_de_semana       int64  
 4   hora_salida         int64  
 5   minuto_salida       int64  
 6   distancia_millas    float64
 7   tiempo_programado   float64
 8   aerolinea           object 
 9   aeropuerto_origen   object 
 10  aeropuerto_destino  object 
 11  estado_origen       object 
 12  estado_destino      object 
 13  vuelo_retrasado     int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 744.0+ MB


In [ ]:
X_input = cleaned_database.drop(columns=['vuelo_retrasado'])
y_true = cleaned_database['vuelo_retrasado']

In [ ]:
import pandas as pd

print(f"Aeropuertos únicos: {cleaned_database['aeropuerto_origen'].nunique()}")


import feature_engineering_functions as func
print(dir(func))

Aeropuertos únicos: 348
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calcular_distancia', 'extraer_features_fecha', 'np', 'pd']


In [ ]:
entrada = pd.DataFrame(data, index=[0])
entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])


entrada_procesada = func.transformar_features(entrada)


custom_model.predict(entrada_procesada)

In [ ]:
def explain(self, X):
    X_feat = self._transform_until_preprocess(X)
    X_trans = self.prep.transform(X_feat)
    shap_values = self.explainer.shap_values(X_trans)


# IMPORTACIÓN - INICIO RÁPIDO

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import precision_recall_curve

print(">>> Cargando modelo y datos...")
model_pipeline = joblib.load('modelo_XGB.joblib')
df_val = pd.read_parquet("df_clean_modeling.parquet")

print(">>> Reconstruyendo fechas (Usando 2024 para soportar bisiestos)...")

df_val['year'] = 2024

fechas_base = pd.to_datetime(
    df_val[['year', 'mes', 'dia_mes']].rename(columns={'mes': 'month', 'dia_mes': 'day'}),
    errors='coerce'
)

df_val['fecha_vuelo'] = fechas_base + \
                        pd.to_timedelta(df_val['hora_salida'], unit='h') + \
                        pd.to_timedelta(df_val['minuto_salida'], unit='m')

nulos = df_val['fecha_vuelo'].isna().sum()
if nulos > 0:
    print(f"⚠️ ADVERTENCIA: {nulos} filas tienen fechas inválidas y serán eliminadas para evitar errores.")
    df_val = df_val.dropna(subset=['fecha_vuelo'])

cols_required = ['aerolinea', 'aeropuerto_origen', 'aeropuerto_destino', 'fecha_vuelo']
X_val = df_val[cols_required].copy()
y_true = df_val['vuelo_retrasado'].values

print(f">>> Ejecutando inferencia en {len(X_val)} vuelos...")
y_proba = model_pipeline.predict_proba(X_val)[:, 1]

df_risk = pd.DataFrame({'y_true': y_true, 'y_proba': y_proba})
df_risk['Decile'] = pd.qcut(df_risk['y_proba'], 10, labels=False, duplicates='drop') + 1

risk_table = df_risk.groupby('Decile').agg(
    Total_Vuelos=('y_true', 'count'),
    Retrasos_Reales=('y_true', 'sum'),
    Tasa_Riesgo=('y_true', 'mean')
).sort_index(ascending=False)

risk_table['Lift'] = risk_table['Tasa_Riesgo'] / df_risk['y_true'].mean()

print("\n=== TABLA DE VALIDACIÓN (LIFT) ===")
print(risk_table)

precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
numerator = 2 * (precision[:-1] * recall[:-1])
denominator = precision[:-1] + recall[:-1]

# Manejo seguro de división por cero
f1_scores = np.divide(numerator, denominator, out=np.zeros_like(denominator), where=denominator!=0)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\n>>> Mejor Umbral: {best_threshold:.4f}")
print(f">>> F1-Score Máximo: {f1_scores[best_idx]:.4f}")

>>> Cargando modelo y datos...
>>> Reconstruyendo fechas (Usando 2024 para soportar bisiestos)...
>>> Ejecutando inferencia en 6965266 vuelos...

=== TABLA DE VALIDACIÓN (LIFT) ===
        Total_Vuelos  Retrasos_Reales  Tasa_Riesgo      Lift
Decile                                                      
10            696527           327272     0.469863  2.356362
9             696525           231610     0.332522  1.667599
8             696526           185435     0.266228  1.335136
7             696527           154408     0.221683  1.111739
6             696520           129372     0.185741  0.931489
5             696534           108123     0.155230  0.778479
4             696524            90037     0.129266  0.648270
3             696529            73237     0.105146  0.527306
2             696526            55119     0.079134  0.396858
1             696528            34273     0.049205  0.246766

>>> Mejor Umbral: 0.5196
>>> F1-Score Máximo: 0.4313


# Función de calculo de riesgos

In [ ]:
import numpy as np
import pandas as pd

def aplicar_niveles_riesgo(df, col_proba='y_proba'):
    """
    Aplica una segmentación de 3 niveles basada en los deciles hallados:
    - Alto: Deciles 9-10 (Top 20%) -> Lift > 1.6 (Zona de Acción Prioritaria)
    - Medio: Deciles 6-8 (Middle 30%) -> Lift ~ 1.0 (Zona de Incertidumbre)
    - Bajo: Deciles 1-5 (Bottom 50%) -> Lift < 0.8 (Zona Segura)
    """

    umbral_alto = np.percentile(df[col_proba], 80)
    umbral_medio = np.percentile(df[col_proba], 50)

    print(f"--- Umbrales Dinámicos Hallados ---")
    print(f"corte_alto (P80):  {umbral_alto:.6f}")
    print(f"corte_medio (P50): {umbral_medio:.6f}")

    condiciones = [
        (df[col_proba] >= umbral_alto),                         # Nivel 3: Alto
        (df[col_proba] >= umbral_medio) & (df[col_proba] < umbral_alto), # Nivel 2: Medio
        (df[col_proba] < umbral_medio)                          # Nivel 1: Bajo
    ]

    etiquetas = ['3. Alto Riesgo', '2. Riesgo Medio', '1. Bajo Riesgo']

    df['Categoria_Riesgo'] = np.select(condiciones, etiquetas, default='Indeterminado')

    return df

df_risk = aplicar_niveles_riesgo(df_risk)

print("\n--- Distribución Final de Niveles ---")
print(df_risk['Categoria_Riesgo'].value_counts(normalize=True).sort_index())

--- Umbrales Dinámicos Hallados ---
corte_alto (P80):  0.620867
corte_medio (P50): 0.455504

--- Distribución Final de Niveles ---
Categoria_Riesgo
1. Bajo Riesgo     0.5
2. Riesgo Medio    0.3
3. Alto Riesgo     0.2
Name: proportion, dtype: float64


In [ ]:
# Tomamos los umbrales del dataframe de riesgo que ya tienes en memoria
# Umbral Alto (Inicio del Decil 9 - Top 20%)
CUTOFF_ALTO = np.percentile(df_risk['y_proba'], 80)

# Umbral Medio (Inicio del Decil 6 - Top 50%)
CUTOFF_MEDIO = np.percentile(df_risk['y_proba'], 50)

print(f"--- PARÁMETROS DE DESPLIEGUE ---")
print(f"Corte Alto Riesgo (>P80):  {CUTOFF_ALTO:.4f}")
print(f"Corte Riesgo Medio (>P50): {CUTOFF_MEDIO:.4f}")

--- PARÁMETROS DE DESPLIEGUE ---
Corte Alto Riesgo (>P80):  0.6209
Corte Riesgo Medio (>P50): 0.4555


In [ ]:
def predecir_riesgo_vuelo(datos_vuelo, modelo, corte_alto, corte_medio):
    """
    1. Recibe datos crudos (Diccionario).
    2. Calcula probabilidad con el modelo.
    3. Asigna etiqueta basada en los cortes pre-calculados.
    """
    df_input = pd.DataFrame([datos_vuelo])
    df_input["fecha_vuelo"] = pd.to_datetime(df_input["fecha_vuelo"])

    probabilidad = modelo.predict_proba(df_input)[0, 1]

    if probabilidad >= corte_alto:
        nivel = "🔴 ALTO"
        accion = "Activar Contingencia / Priorizar Reasignación"
    elif probabilidad >= corte_medio:
        nivel = "🟡 MEDIO"
        accion = "Monitoreo Activo"
    else:
        nivel = "🟢 BAJO"
        accion = "Operación Estándar"

    return {
        "Vuelo": f"{datos_vuelo['aerolinea']} -> {datos_vuelo['aeropuerto_destino']}",
        "Score_Probabilidad": f"{probabilidad:.4f}",
        "Nivel_Riesgo": nivel,
        "Acción_Recomendada": accion
    }

# --- EJEMPLO DE USO ---

# Caso 1: Un vuelo hipotético
vuelo_ejemplo = {
    "aerolinea": "AA",
    "aeropuerto_origen": "JFK",
    "aeropuerto_destino": "LAX",
    "fecha_vuelo": "2024-12-24 18:30" # Nochebuena, alta congestión probable
}


resultado = predecir_riesgo_vuelo(vuelo_ejemplo, model_pipeline, CUTOFF_ALTO, CUTOFF_MEDIO)


for k, v in resultado.items():
    print(f"{k}: {v}")

Vuelo: AA -> LAX
Score_Probabilidad: 0.4049
Nivel_Riesgo: 🟢 BAJO
Acción_Recomendada: Operación Estándar
